# Pipeline Phân Tích Ảnh MRI Não với 3 Mô Hình
## Bước 1: Tải dữ liệu bệnh nhân đầu tiên
## Bước 2: Tiền xử lý dữ liệu
## Bước 3: Chạy thử 3 mô hình (UNet++, DeepLabV3+, SwinUNet)
## Bước 4: Đánh giá theo các chỉ số (Dice, IoU, HD95, Sensitivity, Specificity)

In [1]:
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import os
from pathlib import Path

# Cấu hình thiết bị
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Thiết bị: {DEVICE}")

# Cấu hình đường dẫn
DATA_ROOT = "data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
PATIENT_ID = "BraTS20_Training_001"
PATIENT_PATH = os.path.join(DATA_ROOT, PATIENT_ID)

print(f"\n✓ Đường dẫn dữ liệu: {PATIENT_PATH}")
print(f"✓ Files có sẵn:")
for file in sorted(os.listdir(PATIENT_PATH)):
    print(f"  - {file}")

✓ Thiết bị: cpu

✓ Đường dẫn dữ liệu: data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData\BraTS20_Training_001
✓ Files có sẵn:
  - BraTS20_Training_001_flair.nii
  - BraTS20_Training_001_seg.nii
  - BraTS20_Training_001_t1.nii
  - BraTS20_Training_001_t1ce.nii
  - BraTS20_Training_001_t2.nii


## BƯỚC 1: TIỀN XỬ LÝ DỮ LIỆU

In [2]:
def load_patient_data(patient_path):
    """
    Tải 4 modality MRI của bệnh nhân: T1, T1CE, T2, FLAIR
    """
    modalities = ['t1', 't1ce', 't2', 'flair']
    images = {}
    
    for mod in modalities:
        file_path = os.path.join(patient_path, f"{os.path.basename(patient_path)}_{mod}.nii")
        img = nib.load(file_path).get_fdata()
        images[mod] = img
        print(f"  ✓ {mod.upper()}: {img.shape}")
    
    # Tải segmentation mask
    seg_path = os.path.join(patient_path, f"{os.path.basename(patient_path)}_seg.nii")
    seg = nib.load(seg_path).get_fdata()
    print(f"  ✓ Segmentation: {seg.shape}")
    
    return images, seg

print("Loading data bệnh nhân...")
images, seg_mask = load_patient_data(PATIENT_PATH)

Loading data bệnh nhân...
  ✓ T1: (240, 240, 155)
  ✓ T1CE: (240, 240, 155)
  ✓ T2: (240, 240, 155)
  ✓ FLAIR: (240, 240, 155)
  ✓ Segmentation: (240, 240, 155)


In [3]:
def preprocess_slice(slice_data, target_size=(256, 256)):
    """
    Tiền xử lý 1 lát cắt MRI:
    1. Chuẩn hóa Percentile (1% - 99%)
    2. Resize về 256x256
    """
    # Tạo mask vùng não (non-zero)
    mask = slice_data > 0
    
    if mask.sum() == 0:
        return np.zeros(target_size)
    
    # Percentile normalization
    p1, p99 = np.percentile(slice_data[mask], [1, 99])
    slice_data = np.clip(slice_data, p1, p99)
    slice_data = (slice_data - p1) / (p99 - p1 + 1e-8)
    slice_data[~mask] = 0
    
    # Resize
    slice_resized = cv2.resize(slice_data, target_size, interpolation=cv2.INTER_LINEAR)
    
    return slice_resized

# Chọn lát cắt giữa (depth dimension)
depth = images['t1'].shape[2]
slice_idx = depth // 2

print(f"\n✓ Chọn lát cắt số {slice_idx} (tổng {depth} lát)")

# Tiền xử lý từng modality
preprocessed_slices = []
for mod in ['t1', 't1ce', 't2', 'flair']:
    slice_2d = images[mod][:, :, slice_idx]
    processed = preprocess_slice(slice_2d)
    preprocessed_slices.append(processed)
    print(f"  ✓ Đã xử lý {mod.upper()}")

# Stack 4 channels
input_data = np.stack(preprocessed_slices, axis=-1)  # (256, 256, 4)
print(f"\n✓ Input data shape: {input_data.shape}")


✓ Chọn lát cắt số 77 (tổng 155 lát)
  ✓ Đã xử lý T1
  ✓ Đã xử lý T1CE
  ✓ Đã xử lý T2
  ✓ Đã xử lý FLAIR

✓ Input data shape: (256, 256, 4)


In [4]:
# Tiền xử lý segmentation mask
seg_slice = seg_mask[:, :, slice_idx]
seg_processed = cv2.resize(seg_slice, (256, 256), interpolation=cv2.INTER_NEAREST)

print(f"✓ Segmentation mask shape: {seg_processed.shape}")
print(f"✓ Classes trong mask: {np.unique(seg_processed)}")

# Chuyển sang tensor
X = torch.from_numpy(input_data).permute(2, 0, 1).unsqueeze(0).float()  # (1, 4, 256, 256)
y = torch.from_numpy(seg_processed).unsqueeze(0).unsqueeze(0).long()  # (1, 1, 256, 256)

print(f"\n✓ X tensor: {X.shape} | min={X.min():.4f}, max={X.max():.4f}")
print(f"✓ y tensor: {y.shape} | classes={torch.unique(y).tolist()}")

✓ Segmentation mask shape: (256, 256)
✓ Classes trong mask: [0. 1. 2. 4.]

✓ X tensor: torch.Size([1, 4, 256, 256]) | min=0.0000, max=1.0000
✓ y tensor: torch.Size([1, 1, 256, 256]) | classes=[0, 1, 2, 4]


## BƯỚC 2: ĐỊNH NGHĨA CÁC MÔ HÌNH

In [20]:
import segmentation_models_pytorch as smp

# Cấu hình chung
IN_CHANNELS = 4  # T1, T1CE, T2, FLAIR
NUM_CLASSES = 4  # 0=background, 1=necrotic, 2=edema, 3=enhancing

print("Khởi tạo 3 mô hình...\n")

# Model 1: UNet++
print("1. UNet++ (ResNet34 encoder)")
model_unetpp = smp.UnetPlusPlus(
    encoder_name="resnet34",
    in_channels=IN_CHANNELS,
    classes=NUM_CLASSES,
    activation="softmax",
    decoder_use_batchnorm=True
).to(DEVICE)
print(f"   ✓ Parameters: {sum(p.numel() for p in model_unetpp.parameters()):,}\n")

# Model 2: DeepLabV3+
print("2. DeepLabV3+ (ResNet34 encoder)")
model_deeplabv3 = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    in_channels=IN_CHANNELS,
    classes=NUM_CLASSES,
    activation="softmax"
).to(DEVICE)
print(f"   ✓ Parameters: {sum(p.numel() for p in model_deeplabv3.parameters()):,}\n")

# Model 3: UNet (EfficientNet-b0 encoder) - thay thế SwinUNETR
print("3. UNet (EfficientNet-b0 encoder)")
try:
    model_unet = smp.Unet(
        encoder_name="efficientnet-b0",
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
        activation="softmax"
    ).to(DEVICE)
    print(f"   ✓ Parameters: {sum(p.numel() for p in model_unet.parameters()):,}\n")
except:
    print("   ⚠ EfficientNet không có sẵn, dùng ResNet34 thay thế")
    model_unet = smp.Unet(
        encoder_name="resnet34",
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
        activation="softmax"
    ).to(DEVICE)
    print(f"   ✓ Parameters: {sum(p.numel() for p in model_unet.parameters()):,}\n")

models = {
    'UNet++': model_unetpp,
    'DeepLabV3+': model_deeplabv3,
    'UNet-EfficientNet': model_unet
}

Khởi tạo 3 mô hình...

1. UNet++ (ResNet34 encoder)
   ✓ Parameters: 26,082,180

2. DeepLabV3+ (ResNet34 encoder)
   ✓ Parameters: 22,441,364

3. UNet (EfficientNet-b0 encoder)


c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--smp-hub--efficientnet-b0.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fa

   ✓ Parameters: 6,252,192



## BƯỚC 3: CHẠY THỬ 3 MÔ HÌNH

In [22]:
# Chạy forward pass
X_gpu = X.to(DEVICE)

print("Forward pass cho 3 mô hình...\n")

predictions = {}

with torch.no_grad():
    for name, model in models.items():
        model.eval()
        output = model(X_gpu)
        predictions[name] = output.cpu()
        
        print(f"✓ {name}")
        print(f"  - Output shape: {output.shape}")
        print(f"  - Output range: [{output.min():.4f}, {output.max():.4f}]")
        
        # Lấy predicted class
        pred_class = torch.argmax(output, dim=1, keepdim=True)
        print(f"  - Classes predicted: {torch.unique(pred_class).tolist()}")
        predictions[name + '_class'] = pred_class
        print()

Forward pass cho 3 mô hình...

✓ UNet++
  - Output shape: torch.Size([1, 4, 256, 256])
  - Output range: [0.0949, 0.5450]
  - Classes predicted: [0, 1, 2, 3]

✓ DeepLabV3+
  - Output shape: torch.Size([1, 4, 256, 256])
  - Output range: [0.2356, 0.2678]
  - Classes predicted: [0]

✓ UNet-EfficientNet
  - Output shape: torch.Size([1, 4, 256, 256])
  - Output range: [0.0000, 1.0000]
  - Classes predicted: [0, 1, 2, 3]



## BƯỚC 4: ĐÁNH GIÁ CÁC CHỈ SỐ

In [23]:
from monai.metrics import DiceMetric
from sklearn.metrics import jaccard_score
import numpy as np
import pandas as pd

def evaluate_metrics(y_pred, y_true):
    """
    Tính toán các chỉ số đánh giá:
    - Dice Score
    - IoU (Jaccard)
    - Sensitivity (Recall)
    - Specificity
    - Accuracy
    """
    # Ensure tensors on CPU
    if y_pred.is_cuda:
        y_pred = y_pred.cpu()
    if y_true.is_cuda:
        y_true = y_true.cpu()
    
    # Convert to numpy
    if len(y_pred.shape) > 2:
        y_pred_np = y_pred.argmax(dim=1).numpy().flatten()
    else:
        y_pred_np = y_pred.numpy().flatten()
    
    y_true_np = y_true.numpy().flatten()
    
    # Dice Score
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    try:
        dice = dice_metric(y_pred.float(), y_true.float())
        dice_value = float(dice.item())
    except:
        dice_value = 0.0
    
    # IoU (Jaccard)
    try:
        iou = jaccard_score(y_true_np, y_pred_np, average='weighted', zero_division=0)
    except:
        iou = 0.0
    
    # Accuracy
    accuracy = float((y_pred_np == y_true_np).sum()) / len(y_true_np)
    
    # Sensitivity & Specificity (binary: class 0 vs rest)
    sensitivity = 0.0
    specificity = 0.0
    try:
        y_true_binary = (y_true_np > 0).astype(int)
        y_pred_binary = (y_pred_np > 0).astype(int)
        
        tn = np.sum((y_true_binary == 0) & (y_pred_binary == 0))
        fp = np.sum((y_true_binary == 0) & (y_pred_binary == 1))
        fn = np.sum((y_true_binary == 1) & (y_pred_binary == 0))
        tp = np.sum((y_true_binary == 1) & (y_pred_binary == 1))
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    except:
        pass
    
    return {
        'Dice': dice_value,
        'IoU': iou,
        'Accuracy': accuracy,
        'Sensitivity': sensitivity,
        'Specificity': specificity
    }

print("="*70)
print("KẾT QUẢ ĐÁNH GIÁ CÁC MÔ HÌNH")
print("="*70)

results = {}
for name, model in models.items():
    # Kiểm tra xem prediction có trong dict không
    if name not in predictions:
        print(f"⚠ {name} - không tìm thấy prediction")
        continue
    
    metrics = evaluate_metrics(predictions[name], y)
    results[name] = metrics
    
    print(f"\n{name}:")
    print(f"  Dice Score:    {metrics['Dice']:.4f}")
    print(f"  IoU (Jaccard): {metrics['IoU']:.4f}")
    print(f"  Accuracy:      {metrics['Accuracy']:.4f}")
    print(f"  Sensitivity:   {metrics['Sensitivity']:.4f}")
    print(f"  Specificity:   {metrics['Specificity']:.4f}")

# Bảng so sánh
print("\n" + "="*70)
print("BẢNG SO SÁNH CÁC MÔ HÌNH")
print("="*70)
if results:
    df_results = pd.DataFrame(results).T
    print(df_results.to_string())
    print("\n✓ Mô hình tốt nhất (Dice):")
    best_model = df_results['Dice'].idxmax()
    print(f"  → {best_model} với Dice={df_results.loc[best_model, 'Dice']:.4f}")
else:
    print("⚠ Không có kết quả để hiển thị!")

KẾT QUẢ ĐÁNH GIÁ CÁC MÔ HÌNH

UNet++:
  Dice Score:    0.0000
  IoU (Jaccard): 0.1131
  Accuracy:      0.1245
  Sensitivity:   0.9370
  Specificity:   0.1213

DeepLabV3+:
  Dice Score:    0.0000
  IoU (Jaccard): 0.8518
  Accuracy:      0.9229
  Sensitivity:   0.0000
  Specificity:   1.0000

UNet-EfficientNet:
  Dice Score:    0.0000
  IoU (Jaccard): 0.0105
  Accuracy:      0.0711
  Sensitivity:   0.9935
  Specificity:   0.0053

BẢNG SO SÁNH CÁC MÔ HÌNH
                   Dice       IoU  Accuracy  Sensitivity  Specificity
UNet++              0.0  0.113088  0.124451     0.937030     0.121284
DeepLabV3+          0.0  0.851824  0.922943     0.000000     1.000000
UNet-EfficientNet   0.0  0.010503  0.071091     0.993465     0.005257

✓ Mô hình tốt nhất (Dice):
  → UNet++ với Dice=0.0000


In [24]:
# Bảng so sánh
import pandas as pd

df_results = pd.DataFrame(results).T
print("\n" + "="*70)
print("BẢNG SO SÁNH CÁC MÔ HÌNH")
print("="*70)
print(df_results.to_string())
print("\n✓ Mô hình tốt nhất (Dice):")
best_model = df_results['Dice'].idxmax()
print(f"  → {best_model} với Dice={df_results.loc[best_model, 'Dice']:.4f}")


BẢNG SO SÁNH CÁC MÔ HÌNH
                   Dice       IoU  Accuracy  Sensitivity  Specificity
UNet++              0.0  0.113088  0.124451     0.937030     0.121284
DeepLabV3+          0.0  0.851824  0.922943     0.000000     1.000000
UNet-EfficientNet   0.0  0.010503  0.071091     0.993465     0.005257

✓ Mô hình tốt nhất (Dice):
  → UNet++ với Dice=0.0000
